# SkinDiseaseAI — Phase 2
Acquire the official noncommercial research data, audit images, freeze grouped splits, and verify CPU batches. No model training occurs. This notebook is provided for Colab; it has not been executed in Colab by the agent. Read `docs/DATASET_SOURCE.md` first. Copy the committed repository source to a private Google Drive folder named `SkinDiseaseAI` before running. Do not upload `.venv`, credentials, private chat archives, or user images.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json
PROJECT = Path('/content/drive/MyDrive/SkinDiseaseAI')
assert (PROJECT / 'ai_model/data/pipeline.py').is_file(), 'Copy the repository to PROJECT or edit that path first.'
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('Project:', PROJECT)
print('Python:', sys.version)


## Check the runtime before touching data
Use Colab's available matched PyTorch/Torchvision environment. Do not install the Windows CPU requirements here. If imports or the transform check fail, stop and resolve compatibility; do not continue through a failing checkpoint. A GPU is not required for Phase 2.


In [ ]:
import torch, torchvision, numpy, PIL
from PIL import Image
from ai_model.data.preprocessing import build_transform
print({'torch': torch.__version__, 'torchvision': torchvision.__version__, 'numpy': numpy.__version__, 'Pillow': PIL.__version__, 'cuda': torch.cuda.is_available()})
probe = build_transform()(Image.new('RGB', (200, 150), (124, 116, 104)))
assert probe.shape == (3, 224, 224) and torch.isfinite(probe).all()
print('Shared transform check passed; the generated image above is a technical probe, not research data.')


## Acquire the original data
Images/archive use a temporary runtime cache; reports/manifests will be saved to Drive. The original download includes roughly 2.6 GiB of images. Re-run acquisition after runtime loss. Review the source's CC BY-NC 4.0 attribution requirements. Interrupted downloads can resume while their partial files survive.


In [ ]:
DATA = Path('/content/skindiseaseai-data/ham10000')
OUTPUT = PROJECT / 'data/ham10000/processed-colab-v1'
subprocess.run([sys.executable, '-u', '-m', 'ai_model.data.acquire', '--data-dir', str(DATA), '--extract', '--workers', '4'], check=True)


In [ ]:
subprocess.run([sys.executable, '-u', '-m', 'ai_model.data.pipeline', 'prepare', '--data-dir', str(DATA), '--output', str(OUTPUT)], check=True)
audit = json.loads((OUTPUT / 'audit.json').read_text())
assert audit['passed'] and audit['audited_images'] == 10015
print(json.dumps(json.loads((OUTPUT / 'splits/split_summary.json').read_text()), indent=2))
print(json.dumps(json.loads((OUTPUT / 'loader_verification.json').read_text()), indent=2))


In [ ]:
import shutil
shutil.copy2(DATA / 'provenance.json', OUTPUT / 'source_provenance.json')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(OUTPUT / 'colab-package-freeze.txt').write_text(freeze.stdout)
print('Saved audit, frozen manifests, verification and runtime provenance to:', OUTPUT)
print('Stop here. Ask for Phase 3 approval before training the CNN baseline.')
